# 第10章：大语言模型与智能体（下）

在同一预训练权重上观察LoRA、SFT与DPO；工具与检索采用明确的模拟接口。先运行上篇，生成同目录的nanogpt_pretrained.pt。

从本目录顺序执行。基础依赖为PyTorch、NumPy、Matplotlib，下篇词面检索还使用scikit-learn。

In [ ]:
import sys
from pathlib import Path
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().parent))
torch.set_num_threads(1)
plt.rcParams.update({'font.size': 10, 'pdf.fonttype': 42})

In [ ]:
from nndl.llm import NanoGPT, generate
device = torch.device('cpu')
path = Path('nanogpt_pretrained.pt')
if not path.exists():
    raise FileNotFoundError('请先运行上篇并保存 nanogpt_pretrained.pt')
checkpoint = torch.load(path, map_location=device, weights_only=True)
chars = checkpoint['chars']; vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = dict(enumerate(chars))
def encode(text):
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)
def decode(ids):
    return ''.join(itos[int(i)] for i in ids)

## LoRA：冻结基模型，训练增量

B初始为0，所以首次前向等同于底模；此时A的首次梯度为0，B通常有梯度。适配器继承底模设备和精度。训练参数减少不等于所有内存都按比例减少。

In [ ]:
class LoRALinear(nn.Module):
    """包装一个 nn.Linear，加上 LoRA 增量 (rank-r) 适配器。"""

    def __init__(self, base: nn.Linear, r=4, alpha=16):
        super().__init__()
        if not isinstance(r, int) or r <= 0:
            raise ValueError('LoRA秩须为正整数')
        self.base = base
        self.base.requires_grad_(False)
        self.r = r
        self.scaling = alpha / r
        in_f, out_f = base.in_features, base.out_features
        self.A = nn.Parameter(base.weight.new_zeros(r, in_f))
        self.B = nn.Parameter(base.weight.new_zeros(out_f, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        # B 保持全 0：训练开始时 BA = 0，等价于原模型

    def forward(self, x):
        return self.base(x) + (x @ self.A.t() @ self.B.t()) * self.scaling

In [ ]:
base = nn.Linear(128, 128, bias=False)
adapted = LoRALinear(base, r=4)
print('frozen base:', sum(p.numel() for p in base.parameters()),
      'trainable adapter:', sum(p.numel() for p in adapted.parameters() if p.requires_grad))

### 参数量算例

这是原始Llama-7B量级的Q/K/V/O投影参数算例，不加载该模型，也不是峰值显存测量。

In [ ]:
# Llama-7B 的近似规格
d_model = 4096; n_layers = 32; vocab = 32000
# 全量微调：所有参数都要算梯度
full_finetune_params = 7e9
# LoRA: 通常只对 attention 的 Q/K/V/O 4 个矩阵加 LoRA
# 每层 4 个矩阵，每个矩阵参数量 = 2 * d * r
lora_r = 8
lora_params = n_layers * 4 * 2 * d_model * lora_r       # 8.4M

print(f'Llama-7B 全量微调参数: {full_finetune_params/1e9:.1f}B')
print(f'LoRA-8 训练参数      : {lora_params/1e6:.2f}M')
print(f'LoRA / Full = {lora_params/full_finetune_params*100:.3f}%')

## 加载同一预训练结构并注入LoRA

加载上篇产出的完整配置与词表，去掉微调阶段暂退、冻结底模、精确匹配qkv/proj后注入。上下篇使用相同NanoGPT类，不再重建另一个近似版本。

In [ ]:
def apply_lora_to_model(model, r=8, alpha=16, target_names=('qkv', 'proj')):
    """精确匹配叶模块名；避免再次注入导致嵌套适配器。"""
    if any(isinstance(m, LoRALinear) for m in model.modules()):
        raise ValueError('模型已经含有LoRA；请从未注入的副本开始')
    matches = [(name, m) for name, m in model.named_modules()
               if isinstance(m, nn.Linear) and name.split('.')[-1] in target_names]
    if not matches:
        raise ValueError('没有匹配的线性层，请核对模块名称')
    for name, module in matches:
        parent = model
        *path, leaf = name.split('.')
        for part in path:
            parent = getattr(parent, part)
        setattr(parent, leaf, LoRALinear(module, r=r, alpha=alpha))
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'可训练 {n_train:,} / 总 {n_total:,} ({n_train/n_total*100:.2f}%)')
    return model

# 重新加载选定预训练快照；微调时把所有暂退率设为0，便于小实验复核。
checkpoint = torch.load('nanogpt_pretrained.pt', map_location=device, weights_only=True)
assert checkpoint['chars'] == chars
sft_config = dict(checkpoint['config'], dropout=0.0)
model = NanoGPT(**sft_config).to(device)
model.load_state_dict(checkpoint['model'])
model.requires_grad_(False)
torch.manual_seed(300)
apply_lora_to_model(model, r=8, alpha=16)

### SFT回答掩码与补齐

最后一个提示位置监督首个回答词元；右侧补齐目标为−100。保留整个提示，回答可以按窗口截短；无回答空间则报错。该边界处理针对字符词元化，换子词或聊天模板时需重新核验。

In [ ]:
def build_sft_batch(pairs, encode_fn=encode, max_len=64):
    """max_len是输入长度上限；保留整个提示，回答最多保留剩余位置。"""
    if not pairs or max_len < 1:
        raise ValueError('批次非空，max_len须为正')
    xs, ys = [], []
    for prompt, response in pairs:
        p_ids = encode_fn(prompt).tolist()
        r_ids = encode_fn(response).tolist()
        if not p_ids or not r_ids:
            raise ValueError('本例要求非空提示和回答')
        if len(p_ids) > max_len:
            raise ValueError('提示占满上下文，无法监督回答；请缩短提示')
        r_ids = r_ids[:max_len + 1 - len(p_ids)]
        ids = torch.tensor(p_ids + r_ids, dtype=torch.long)
        x, y = ids[:-1], ids[1:].clone()
        y[:len(p_ids)-1] = -100   # 最后一个提示位置预测第一个回答词元
        xs.append(x); ys.append(y)
    from torch.nn.utils.rnn import pad_sequence
    x = pad_sequence(xs, batch_first=True, padding_value=0)
    y = pad_sequence(ys, batch_first=True, padding_value=-100)
    return x, y

## 大小写转换：训练与保留单词

固定14个训练词、6个验证词、6个测试词，200次更新，用验证回答损失选择模型。训练词完全匹配是记忆检查，保留测试词才观察词面泛化。业务上的大小写转换直接调用字符串函数即可。

In [ ]:
# 固定单词划分：同一单词不跨训练、验证、测试集合。
WORDS = ['hello', 'world', 'pytorch', 'transformer', 'language', 'model',
         'small', 'cat', 'dog', 'machine', 'learn', 'deep', 'neural', 'net']
VAL_WORDS = ['book', 'river', 'music', 'dream', 'school', 'garden']
TEST_WORDS = ['attention', 'tensor', 'network', 'science', 'paper', 'graph']

def make_pair(word):
    return f'Convert: {word}: ', word.upper() + '\n'

@torch.no_grad()
def sft_loss_on(words):
    was_training = model.training; model.eval()
    try:
        x, y = build_sft_batch([make_pair(w) for w in words], max_len=model.block_size)
        _, loss = model(x.to(device), y.to(device))
        return loss.item()
    finally:
        model.train(was_training)

sft_generator = torch.Generator().manual_seed(300)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=1e-3, weight_decay=0.0)
sft_hist = {'step': [], 'train_loss': [], 'val_loss': []}
sft_best_loss, sft_best_state, sft_best_step = float('inf'), None, 0
for step in range(201):
    if step > 0:
        model.train()
        indices = torch.randint(len(WORDS), (8,), generator=sft_generator)
        x, y = build_sft_batch([make_pair(WORDS[int(i)]) for i in indices], max_len=model.block_size)
        _, loss = model(x.to(device), y.to(device))
        opt.zero_grad(); loss.backward(); opt.step()
    if step % 20 == 0:
        train_loss, val_loss = sft_loss_on(WORDS), sft_loss_on(VAL_WORDS)
        sft_hist['step'].append(step); sft_hist['train_loss'].append(train_loss)
        sft_hist['val_loss'].append(val_loss)
        print(f'SFT step={step:3d} train={train_loss:.4f} val={val_loss:.4f}')
        if val_loss < sft_best_loss:
            sft_best_loss, sft_best_step = val_loss, step
            sft_best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(sft_best_state)

@torch.no_grad()
def conversion_results(words):
    rows = []
    for word in words:
        prompt, target = make_pair(word)
        ids = encode(prompt).unsqueeze(0)
        out = generate(model, ids, max_new_tokens=32, temperature=0, stop_id=stoi['\n'])
        response = decode(out[0, ids.shape[1]:].tolist())
        rows.append({'word': word, 'response': response, 'correct': response == target})
    return rows

sft_train_predictions = conversion_results(WORDS)
sft_test_predictions = conversion_results(TEST_WORDS)
sft_test_loss = sft_loss_on(TEST_WORDS)
print('SFT best_step:', sft_best_step)
print('训练词完全匹配：', sum(x['correct'] for x in sft_train_predictions), '/', len(WORDS))
print('保留测试词完全匹配：', sum(x['correct'] for x in sft_test_predictions), '/', len(TEST_WORDS))
for row in sft_test_predictions:
    print(row['word'], '->', repr(row['response']))
torch.save({'config': sft_config, 'chars': chars, 'model': sft_best_state,
            'lora': {'r': 8, 'alpha': 16}}, 'nanogpt_sft.pt')

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 2.8))
ax.plot(sft_hist['step'], sft_hist['train_loss'], label='training words')
ax.plot(sft_hist['step'], sft_hist['val_loss'], label='validation words')
ax.set_xlabel('updates'); ax.set_ylabel('response-token cross entropy')
ax.legend(); ax.grid(alpha=.2); fig.tight_layout(); plt.show()

## DPO：固定参考的偏好比较

教学偏好由规则产生：大写回答优于小写回答。相同策略/参考初始化、关闭暂退时，损失应为ln(2)，仍有策略梯度。参考模型冻结，偏好损失下降不保证胜出回答的绝对概率增加。

In [ ]:
def logprob_of_sequence(model, prompt, response):
    """字符级条件对数概率之和；不允许截断改变偏好样本。"""
    if len(encode(prompt)) + len(encode(response)) - 1 > model.block_size:
        raise ValueError('偏好样本超出上下文，应在数据准备时统一处理')
    x, y = build_sft_batch([(prompt, response)], max_len=model.block_size)
    model_device = next(model.parameters()).device
    x, y = x.to(model_device), y.to(model_device)
    logits, _ = model(x)
    token_logps = F.log_softmax(logits, dim=-1).gather(-1, y.clamp_min(0).unsqueeze(-1)).squeeze(-1)
    return token_logps.masked_select(y != -100).sum()

def dpo_loss(policy, ref, prompt, y_win, y_lose, beta=0.1):
    if beta <= 0:
        raise ValueError('beta须为正')
    lp_w = logprob_of_sequence(policy, prompt, y_win)
    lp_l = logprob_of_sequence(policy, prompt, y_lose)
    with torch.no_grad():
        lr_w = logprob_of_sequence(ref, prompt, y_win)
        lr_l = logprob_of_sequence(ref, prompt, y_lose)
    return -F.logsigmoid(beta * ((lp_w - lr_w) - (lp_l - lr_l)))

# 教学偏好：同一词的大写回答胜过小写回答，不声称这是人类标注数据。
import copy
policy = copy.deepcopy(model).eval()
ref = copy.deepcopy(model).requires_grad_(False).eval()
dpo_optimizer = torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad],
                                  lr=1e-4, weight_decay=0)
dpo_generator = torch.Generator().manual_seed(400)

def preference(word):
    prompt, win = make_pair(word)
    return prompt, win, word + '\n'

dpo_hist = []
for step in range(101):
    if step > 0:
        chosen = torch.randint(len(WORDS), (4,), generator=dpo_generator)
        loss = torch.stack([dpo_loss(policy, ref, *preference(WORDS[int(i)])) for i in chosen]).mean()
        dpo_optimizer.zero_grad(); loss.backward(); dpo_optimizer.step()
    if step % 20 == 0:
        with torch.no_grad():
            train_loss = torch.stack([dpo_loss(policy, ref, *preference(w)) for w in WORDS]).mean().item()
            val_loss = torch.stack([dpo_loss(policy, ref, *preference(w)) for w in VAL_WORDS]).mean().item()
        dpo_hist.append({'step': step, 'train_loss': train_loss, 'val_loss': val_loss})
        print(f'DPO step={step:3d} train={train_loss:.4f} val={val_loss:.4f}')
with torch.no_grad():
    dpo_test_loss = torch.stack([dpo_loss(policy, ref, *preference(w)) for w in TEST_WORDS]).mean().item()
print(f'DPO held-out preference loss={dpo_test_loss:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 2.8))
for key in ['train_loss', 'val_loss']:
    ax.plot([r['step'] for r in dpo_hist], [r[key] for r in dpo_hist], label=key)
ax.axhline(math.log(2), color='gray', ls='--', label='identical reference')
ax.set_xlabel('updates'); ax.set_ylabel('preference loss')
ax.legend(); ax.grid(alpha=.2); fig.tight_layout(); plt.show()

## 结构化工具调用与错误状态

mock_llm按固定规则决定动作；不是前面的NanoGPT在使用工具。输入结构、工具白名单、运算范围与步数由程序检查；成功和错误都作为观察返回。Search只查固定离线条目。

In [ ]:
import ast
import operator
import re

def calculator(expression):
    """仅解释有界四则运算，不执行Python表达式中的任意操作。"""
    if not isinstance(expression, str) or not 1 <= len(expression) <= 128:
        raise ValueError('表达式长度须在1至128字符之间')
    tree = ast.parse(expression, mode='eval')
    if len(list(ast.walk(tree))) > 50:
        raise ValueError('表达式太复杂')
    binary = {ast.Add: operator.add, ast.Sub: operator.sub,
              ast.Mult: operator.mul, ast.Div: operator.truediv}
    unary = {ast.UAdd: operator.pos, ast.USub: operator.neg}
    def visit(node, depth=0):
        if depth > 12:
            raise ValueError('表达式嵌套太深')
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            value = node.value
        elif isinstance(node, ast.BinOp) and type(node.op) in binary:
            value = binary[type(node.op)](visit(node.left, depth+1), visit(node.right, depth+1))
        elif isinstance(node, ast.UnaryOp) and type(node.op) in unary:
            value = unary[type(node.op)](visit(node.operand, depth+1))
        else:
            raise ValueError('只支持数字、括号和四则运算')
        if not math.isfinite(value) or abs(value) > 1e12:
            raise ValueError('数值超出教学工具范围')
        return value
    return str(visit(tree.body))

def search(query):
    # 固定的离线资料，并非联网搜索。
    entries = {'法国首都': '巴黎', 'capital of france': 'Paris',
               '真空光速': '299792458 m/s'}
    for key, value in entries.items():
        if key in query.lower():
            return value
    raise LookupError('离线资料中没有相关条目')

TOOLS = {'Calculator': calculator, 'Search': search}

def mock_llm(question, history):
    """按固定规则返回结构化动作，用于检查执行循环。"""
    if history and history[-1]['kind'] == 'observation':
        observation = history[-1]
        answer = observation['result'] if observation['ok'] else '工具未能完成：' + observation['result']
        return {'kind': 'answer', 'text': answer}
    match = re.search(r'[0-9(][0-9+*/(). ×÷-]*', question)
    if match and any(op in match.group() for op in '+-*/×÷'):
        return {'kind': 'tool', 'name': 'Calculator',
                'input': match.group().strip().replace('×', '*').replace('÷', '/')}
    return {'kind': 'tool', 'name': 'Search', 'input': question}

def react_loop(question, max_steps=5, model_fn=mock_llm):
    if max_steps < 1:
        raise ValueError('max_steps须为正')
    history = []
    for _ in range(max_steps):
        action = model_fn(question, history)
        if not isinstance(action, dict):
            return {'status': 'invalid', 'history': history}
        kind = action.get('kind')
        if kind == 'answer' and isinstance(action.get('text'), str):
            history.append(action)
            return {'status': 'answered', 'answer': action['text'], 'history': history}
        if kind != 'tool' or not isinstance(action.get('name'), str) or not isinstance(action.get('input'), str):
            return {'status': 'invalid', 'history': history}
        history.append(action)
        try:
            if action['name'] not in TOOLS:
                raise ValueError('未知工具')
            if len(action['input']) > 256:
                raise ValueError('工具输入过长')
            result = TOOLS[action['name']](action['input'])
            observation = {'kind': 'observation', 'ok': True, 'result': result}
        except (ValueError, SyntaxError, ArithmeticError, LookupError) as error:
            observation = {'kind': 'observation', 'ok': False, 'result': str(error)}
        history.append(observation)
    return {'status': 'limit', 'history': history}

for question in ['计算 (15 + 27) * 3', '法国首都是什么？', '计算 1 / 0']:
    result = react_loop(question)
    print(question, '->', result.get('answer', result['status']))

In [ ]:
# 正常、错误、无资料、结构错误与资源上限应给出可辨认状态。
for question in ['计算 (15 + 27) * 3', '计算 1 / 0', '今天的天气是什么？']:
    print(react_loop(question))
print(react_loop('x', model_fn=lambda q, h: 'wrong format'))
repeating = lambda q, h: {'kind': 'tool', 'name': 'Search', 'input': '法国首都'}
print(react_loop('x', max_steps=2, model_fn=repeating)['status'])

## 带证据来源的检索流程

虚构课程手册用字符TF-IDF作词面检索，回答函数只是带来源的摘录，没有调用真实生成模型。把检索召回、引用支持与最终生成分开检查；没有证据时返回资料不足。

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 教学虚构的课程手册；保留文档ID与来源，便于核对检索结果。
kb_chunks = [
    {'id': 'gpu', 'source': '课程手册第1节', 'text': 'GPU单次预约上限是2小时。取消预约后需要重新提交。'},
    {'id': 'report', 'source': '课程手册第2节', 'text': '实验报告应包括训练曲线、验证指标和数据划分方法。'},
    {'id': 'backup', 'source': '课程手册第3节', 'text': '模型文件保存在checkpoints目录，提交报告前请另存一份备份。'},
]
# 字符n-gram的稀疏向量是词面检索基线，不是预训练语义嵌入。
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), norm='l2')
kb_vectors = vectorizer.fit_transform([chunk['text'] for chunk in kb_chunks])

def retrieve(question, k=2, min_score=0.1):
    if not isinstance(k, int) or not 1 <= k <= len(kb_chunks):
        raise ValueError('k超出知识库大小')
    q = vectorizer.transform([question])
    scores = (kb_vectors @ q.T).toarray().ravel()
    indices = scores.argsort(kind='stable')[::-1][:k]
    return [dict(kb_chunks[i], score=float(scores[i])) for i in indices if scores[i] >= min_score]

def mock_grounded_answer(question, evidence):
    # 用摘录代替真实生成，明确标记来源；不把证据内容当作可执行指令。
    if not evidence:
        return '现有课程手册没有足够资料回答。'
    first = evidence[0]
    return f"资料摘录：{first['text']} [{first['id']}; {first['source']}]"

def rag(question, k=2, answer_fn=mock_grounded_answer):
    evidence = retrieve(question, k)
    return {'answer': answer_fn(question, evidence), 'evidence': evidence}

for question in ['GPU预约上限是多少？', '实验报告应包含什么？', '明天天气如何？']:
    result = rag(question)
    print(question, '->', result['answer'])

In [ ]:
# 微型检索检查：预先标注所需段落，再检查Top-k召回。
retrieval_cases = [('GPU预约上限是多少？', 'gpu'),
                   ('实验报告应包含什么？', 'report'),
                   ('模型文件保存在什么目录？', 'backup')]
hits = 0
for question, expected_id in retrieval_cases:
    evidence = retrieve(question)
    hits += expected_id in {item['id'] for item in evidence}
    print(question, '->', [item['id'] for item in evidence])
print('known-evidence recall:', hits, '/', len(retrieval_cases))
print('unanswerable:', rag('明天天气如何？'))

## 阅读与扩展

离线DPO、可核验奖励的强化学习和推断时增加采样预算各有不同目标。接入真实模型需要适配结构化输出、工具权限与错误状态；检索内容作为资料，不作为操作指令。

参考：[LoRA](https://arxiv.org/abs/2106.09685)、[DPO](https://arxiv.org/abs/2305.18290)、[ReAct](https://arxiv.org/abs/2210.03629)、[RAG](https://arxiv.org/abs/2005.11401)。